# SN 光谱诊断：时间序列、谱线速度、pEW/FWHM、黑体温度

这个 notebook 读取 `data/SN*/SN*_bfosc_*.fits` 这类一维光谱 FITS，并自动做一批分类之外的诊断：

1. 汇总每条光谱的观测时间、目标、仪器、波长范围、流量单位。
2. 从本地 `data/tns_public_objects.csv` 自动补红移、类型、发现日期。
3. 按目标画多历元光谱序列。
4. 按 SN 类型选择常用谱线，测吸收谷速度、pseudo-EW、FWHM 和线深。
5. 粗略拟合连续谱黑体温度。
6. 输出诊断表到 `output/spectral_diagnostics/`。

限制：谱线测量采用自动局部连续谱和吸收谷搜索，适合批量初筛。关键结果需要人工检查线识别和搜索窗口。

In [ ]:
%matplotlib inline

import os
import sys
import csv
import math
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'matplotlib'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from astropy.io import fits
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'spectral_diagnostics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 用户配置

`TARGETS = []` 表示处理所有 `data/SN*/` 下的一维光谱。若某个目标在本地 TNS catalog 没有红移，填到 `TARGET_OVERRIDES`。

In [ ]:
TARGETS = ['SN2026kid']  # 例如 ['SN2026kid', 'SN2026jlm']; 空列表表示全部
AUTO_SAVE = False

# 本地 catalog 缺失或你想覆盖时，在这里填。
TARGET_OVERRIDES = {
    # 'SN2026lmp': {'z': 0.0123, 'type': 'SN Ia', 'discoverydate': '2026-05-01 00:00:00'},
}

# 可按目标手动指定谱线。留空则按 SN 类型自动选择。
TARGET_LINES = {
    # 'SN2026kid': ['Halpha', 'Hbeta', 'FeII5169'],
    # 'SN2026jlm': ['SiII6355', 'CaIIHK', 'SII5640'],
}

# 自动线测量窗口，单位 Angstrom，定义在 rest frame。
LINE_HALF_WIDTH = 420.0

# 黑体拟合波段。过蓝区常有 line blanketing，Ia 的温度只做粗略参考。
BB_WAVE_RANGE = (4200.0, 7600.0)

## 2. 工具函数

In [ ]:
C_KMS = 299792.458

LINE_LIBRARY = {
    'SiII6355': {'rest': 6355.0, 'label': 'Si II 6355', 'blue_only': True},
    'CaIIHK': {'rest': 3933.7, 'label': 'Ca II H&K / K', 'blue_only': True},
    'SII5640': {'rest': 5640.0, 'label': 'S II W approx', 'blue_only': False},
    'Halpha': {'rest': 6562.8, 'label': 'H alpha', 'blue_only': True},
    'Hbeta': {'rest': 4861.3, 'label': 'H beta', 'blue_only': True},
    'FeII5169': {'rest': 5169.0, 'label': 'Fe II 5169', 'blue_only': True},
    'HeI5876': {'rest': 5875.6, 'label': 'He I 5876', 'blue_only': True},
    'HeI6678': {'rest': 6678.2, 'label': 'He I 6678', 'blue_only': True},
    'OI7774': {'rest': 7774.0, 'label': 'O I 7774', 'blue_only': True},
    'CaIINIR': {'rest': 8579.0, 'label': 'Ca II NIR triplet approx', 'blue_only': True},
}


def normalize_target_name(value):
    value = str(value or '').strip().replace(' ', '')
    if not value:
        return ''
    upper = value.upper()
    if upper.startswith(('SN', 'AT')):
        return upper
    if value[:4].isdigit():
        return 'SN' + upper
    return upper


def parse_float(value):
    try:
        if value is None or str(value).strip() in {'', 'None', 'nan', '---'}:
            return np.nan
        return float(value)
    except Exception:
        return np.nan


def parse_datetime(value):
    if value is None or str(value).strip() == '':
        return pd.NaT
    return pd.to_datetime(str(value).replace('T', ' '), errors='coerce', utc=False)


def load_tns_metadata(csv_path):
    metadata = {}
    if not csv_path.exists():
        return metadata
    lines = csv_path.read_text(encoding='utf-8', errors='replace').splitlines()
    start = 1 if lines and not lines[0].startswith('"') else 0
    reader = csv.DictReader(lines[start:])
    for row in reader:
        key = normalize_target_name(f"{row.get('name_prefix','')}{row.get('name','')}")
        if not key:
            continue
        metadata[key] = {
            'z': parse_float(row.get('redshift')),
            'type': row.get('type', ''),
            'discoverydate': row.get('discoverydate', ''),
            'ra': parse_float(row.get('ra')),
            'dec': parse_float(row.get('declination')),
        }
    return metadata


def wavelength_axis_from_header(header, n):
    crval = header.get('CRVAL1')
    crpix = header.get('CRPIX1', 1.0)
    cdelt = header.get('CDELT1', header.get('CD1_1'))
    if crval is None or cdelt is None:
        return None
    pix = np.arange(n, dtype=float) + 1.0
    return float(crval) + (pix - float(crpix)) * float(cdelt)


def odd_window(n, preferred):
    if n < 5:
        return None
    w = min(int(preferred), n if n % 2 == 1 else n - 1)
    if w < 5:
        w = 5 if n >= 5 else None
    if w is not None and w % 2 == 0:
        w -= 1
    return w


def smooth_flux(flux, preferred_window=41):
    flux = np.asarray(flux, dtype=float)
    good = np.isfinite(flux)
    if good.sum() < 7:
        return flux
    filled = flux.copy()
    if not good.all():
        x = np.arange(len(flux))
        filled[~good] = np.interp(x[~good], x[good], flux[good])
    w = odd_window(len(filled), preferred_window)
    if w is None or w <= 3:
        return filled
    return savgol_filter(filled, window_length=w, polyorder=min(3, w - 2))


def default_lines_for_type(sn_type):
    t = str(sn_type or '').lower()
    if 'ia' in t:
        return ['SiII6355', 'CaIIHK', 'SII5640']
    if 'iib' in t:
        return ['Halpha', 'Hbeta', 'HeI5876', 'FeII5169']
    if 'iin' in t or ' ii' in t or t.endswith('ii'):
        return ['Halpha', 'Hbeta', 'FeII5169']
    if 'ib' in t:
        return ['HeI5876', 'HeI6678', 'CaIIHK']
    if 'ic' in t:
        return ['OI7774', 'CaIIHK', 'FeII5169']
    return ['Halpha', 'SiII6355', 'HeI5876']


def local_linear_continuum(wave, flux, edge_fraction=0.18):
    n = len(wave)
    edge = max(4, int(edge_fraction * n))
    left_x = np.nanmedian(wave[:edge])
    right_x = np.nanmedian(wave[-edge:])
    left_y = np.nanmedian(flux[:edge])
    right_y = np.nanmedian(flux[-edge:])
    if not np.isfinite(left_y) or not np.isfinite(right_y) or right_x == left_x:
        return np.full_like(flux, np.nanmedian(flux))
    slope = (right_y - left_y) / (right_x - left_x)
    return left_y + slope * (wave - left_x)


def measure_absorption_line(wave_rest, flux, line_key, half_width=420.0):
    line = LINE_LIBRARY[line_key]
    rest = line['rest']
    mask = (wave_rest > rest - half_width) & (wave_rest < rest + half_width)
    if mask.sum() < 12:
        return {'line': line_key, 'status': 'outside wavelength range'}

    w = np.asarray(wave_rest[mask], dtype=float)
    f = smooth_flux(np.asarray(flux[mask], dtype=float), preferred_window=21)
    cont = local_linear_continuum(w, f)
    valid = np.isfinite(w) & np.isfinite(f) & np.isfinite(cont) & (np.abs(cont) > 0)
    if valid.sum() < 12:
        return {'line': line_key, 'status': 'bad local continuum'}

    w = w[valid]
    norm = f[valid] / cont[valid]
    search = w < rest if line.get('blue_only', True) else np.isfinite(w)
    if search.sum() < 5:
        search = np.isfinite(w)
    idx_candidates = np.where(search)[0]
    min_i = idx_candidates[np.nanargmin(norm[idx_candidates])]
    abs_wave = float(w[min_i])
    min_norm = float(norm[min_i])
    depth = max(0.0, 1.0 - min_norm)
    velocity = C_KMS * (rest - abs_wave) / rest

    absorption = np.clip(1.0 - norm, 0.0, None)
    pew = float(np.trapz(absorption, w))

    fwhm = np.nan
    if depth > 0:
        half_level = 1.0 - depth / 2.0
        below = w[norm <= half_level]
        if len(below) >= 2:
            fwhm = float(below.max() - below.min())

    return {
        'line': line_key,
        'line_label': line['label'],
        'rest_wave': rest,
        'abs_wave': abs_wave,
        'velocity_kms': float(velocity),
        'pEW_A': pew,
        'FWHM_A': fwhm,
        'depth': depth,
        'status': 'ok',
    }


def planck_lambda_angstrom(wave_a, temperature, amplitude):
    wave_m = np.asarray(wave_a, dtype=float) * 1e-10
    h = 6.62607015e-34
    c = 299792458.0
    k = 1.380649e-23
    x = h * c / (wave_m * k * temperature)
    x = np.clip(x, 1e-6, 700)
    b = (2.0 * h * c**2) / (wave_m**5 * (np.exp(x) - 1.0))
    return amplitude * b / np.nanmedian(b)


def fit_blackbody_temperature(wave_rest, flux, wave_range=(4200.0, 7600.0)):
    wave_rest = np.asarray(wave_rest, dtype=float)
    flux = smooth_flux(np.asarray(flux, dtype=float), preferred_window=101)
    mask = (wave_rest > wave_range[0]) & (wave_rest < wave_range[1]) & np.isfinite(flux) & (flux > 0)
    if mask.sum() < 30:
        return {'T_bb_K': np.nan, 'T_err_K': np.nan, 'status': 'not enough positive continuum'}
    x = wave_rest[mask]
    y = flux[mask]
    y = y / np.nanmedian(y)
    try:
        popt, pcov = curve_fit(
            planck_lambda_angstrom, x, y,
            p0=(8000.0, 1.0),
            bounds=([3000.0, 0.01], [25000.0, 100.0]),
            maxfev=20000,
        )
        perr = np.sqrt(np.diag(pcov)) if pcov.size else [np.nan, np.nan]
        return {'T_bb_K': float(popt[0]), 'T_err_K': float(perr[0]), 'status': 'ok'}
    except Exception as exc:
        return {'T_bb_K': np.nan, 'T_err_K': np.nan, 'status': f'fit failed: {exc}'}

## 3. 读取本地 TNS 元数据和 FITS 光谱

In [ ]:
tns_meta = load_tns_metadata(DATA_DIR / 'tns_public_objects.csv')
for target, override in TARGET_OVERRIDES.items():
    key = normalize_target_name(target)
    base = tns_meta.get(key, {})
    base.update(override)
    tns_meta[key] = base

fits_paths = sorted(
    p for p in DATA_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in {'.fits', '.fit', '.fts'}
)

if TARGETS:
    wanted = {normalize_target_name(t) for t in TARGETS}
    fits_paths = [p for p in fits_paths if normalize_target_name(p.parent.name) in wanted or normalize_target_name(p.stem.split('_')[0]) in wanted]

spectra = []
skipped = []
for path in fits_paths:
    try:
        with fits.open(path, memmap=False) as hdul:
            hdu = hdul[0]
            data = hdu.data
            if data is None or np.asarray(data).ndim != 1:
                skipped.append({'file': str(path.relative_to(PROJECT_ROOT)), 'reason': 'not 1D spectrum'})
                continue
            flux = np.asarray(data, dtype=float)
            wave = wavelength_axis_from_header(hdu.header, len(flux))
            if wave is None:
                skipped.append({'file': str(path.relative_to(PROJECT_ROOT)), 'reason': 'no wavelength WCS'})
                continue
            target = normalize_target_name(hdu.header.get('OBJECT') or path.parent.name or path.stem.split('_')[0])
            meta = tns_meta.get(target, {})
            date_obs = parse_datetime(hdu.header.get('DATE-OBS', ''))
            discovery = parse_datetime(meta.get('discoverydate', ''))
            phase_days = np.nan
            if pd.notna(date_obs) and pd.notna(discovery):
                phase_days = (date_obs - discovery).total_seconds() / 86400.0
            spectra.append({
                'path': path,
                'file': str(path.relative_to(PROJECT_ROOT)),
                'target': target,
                'wave': np.asarray(wave, dtype=float),
                'flux': flux,
                'date_obs': date_obs,
                'phase_days': phase_days,
                'z': parse_float(meta.get('z')),
                'type': meta.get('type', ''),
                'discoverydate': meta.get('discoverydate', ''),
                'exptime': parse_float(hdu.header.get('EXPTIME', hdu.header.get('EXPOSURE'))),
                'telescope': hdu.header.get('TELESCOP', ''),
                'instrument': hdu.header.get('INSTRUME', ''),
                'setup': hdu.header.get('FILTER', hdu.header.get('GRISM', '')),
                'bunit': hdu.header.get('BUNIT', ''),
            })
    except Exception as exc:
        skipped.append({'file': str(path.relative_to(PROJECT_ROOT)), 'reason': repr(exc)})

summary_rows = []
for s in spectra:
    f = s['flux']
    w = s['wave']
    finite = np.isfinite(f)
    summary_rows.append({
        'target': s['target'],
        'file': s['file'],
        'date_obs': s['date_obs'],
        'phase_days': s['phase_days'],
        'type': s['type'],
        'z': s['z'],
        'n_pix': len(f),
        'wave_min': np.nanmin(w),
        'wave_max': np.nanmax(w),
        'flux_median': np.nanmedian(f[finite]) if finite.any() else np.nan,
        'exptime': s['exptime'],
        'instrument': s['instrument'],
        'setup': s['setup'],
    })

summary = pd.DataFrame(summary_rows).sort_values(['target', 'date_obs', 'file']).reset_index(drop=True)
display(summary)

if skipped:
    print('Skipped files:')
    display(pd.DataFrame(skipped))

## 4. 多历元光谱序列

每条谱按中位数归一化，并按日期顺序加 offset。

In [ ]:
if not spectra:
    raise RuntimeError('No 1D calibrated spectra found')

for target in sorted({s['target'] for s in spectra}):
    items = sorted([s for s in spectra if s['target'] == target], key=lambda x: (pd.Timestamp.max if pd.isna(x['date_obs']) else x['date_obs']))
    plt.figure(figsize=(11, 1.8 + 1.1 * len(items)))
    for i, s in enumerate(items):
        wave = s['wave']
        flux = smooth_flux(s['flux'], preferred_window=11)
        finite = np.isfinite(flux)
        scale = np.nanmedian(np.abs(flux[finite])) if finite.any() else 1.0
        if not np.isfinite(scale) or scale == 0:
            scale = 1.0
        y = flux / scale + i * 1.5
        phase = s['phase_days']
        phase_label = '' if not np.isfinite(phase) else f', +{phase:.1f} d from discovery'
        date_label = '' if pd.isna(s['date_obs']) else s['date_obs'].strftime('%Y-%m-%d')
        plt.plot(wave, y, lw=0.8, label=f"{date_label}{phase_label}")
    plt.title(f'{target} spectral time series')
    plt.xlabel('Observed wavelength (Angstrom)')
    plt.ylabel('Scaled flux + offset')
    plt.grid(alpha=0.2)
    plt.legend(fontsize=8)
    plt.show()

## 5. 自动谱线诊断

输出每条光谱、每条线的吸收谷速度、pseudo-EW、FWHM 和线深。`velocity_kms` 为正值表示吸收谷在蓝侧。

In [ ]:
line_rows = []
bb_rows = []

for s in spectra:
    target = s['target']
    z = s['z']
    wave_rest = s['wave'] / (1.0 + z) if np.isfinite(z) else s['wave'].copy()
    line_keys = TARGET_LINES.get(target, default_lines_for_type(s['type']))

    bb = fit_blackbody_temperature(wave_rest, s['flux'], BB_WAVE_RANGE)
    bb_rows.append({
        'target': target,
        'file': s['file'],
        'date_obs': s['date_obs'],
        'phase_days': s['phase_days'],
        'type': s['type'],
        'z': z,
        **bb,
    })

    for line_key in line_keys:
        if line_key not in LINE_LIBRARY:
            continue
        result = measure_absorption_line(wave_rest, s['flux'], line_key, LINE_HALF_WIDTH)
        line_rows.append({
            'target': target,
            'file': s['file'],
            'date_obs': s['date_obs'],
            'phase_days': s['phase_days'],
            'type': s['type'],
            'z': z,
            **result,
        })

line_df = pd.DataFrame(line_rows).sort_values(['target', 'line', 'date_obs', 'file']).reset_index(drop=True)
bb_df = pd.DataFrame(bb_rows).sort_values(['target', 'date_obs', 'file']).reset_index(drop=True)

display(line_df)
display(bb_df)

## 6. 速度、pEW、FWHM 随时间演化

In [ ]:
ok = line_df[line_df['status'].eq('ok')].copy()

if ok.empty:
    print('No successful line measurements.')
else:
    for target in sorted(ok['target'].unique()):
        sub = ok[ok['target'] == target]
        xcol = 'phase_days' if sub['phase_days'].notna().any() else 'date_obs'
        fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
        for line, g in sub.groupby('line'):
            g = g.sort_values(xcol)
            x = g[xcol]
            axes[0].plot(x, g['velocity_kms'], marker='o', label=line)
            axes[1].plot(x, g['pEW_A'], marker='o', label=line)
            axes[2].plot(x, g['FWHM_A'], marker='o', label=line)
        axes[0].set_ylabel('Velocity (km/s)')
        axes[1].set_ylabel('pEW (A)')
        axes[2].set_ylabel('FWHM (A)')
        axes[2].set_xlabel('Days since discovery' if xcol == 'phase_days' else 'Obs date')
        axes[0].set_title(f'{target} line diagnostics')
        for ax in axes:
            ax.grid(alpha=0.2)
            ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()

## 7. 连续谱黑体温度粗估

In [ ]:
ok_bb = bb_df[bb_df['status'].eq('ok')].copy()
if ok_bb.empty:
    print('No successful blackbody fits.')
else:
    plt.figure(figsize=(10, 5))
    for target, g in ok_bb.groupby('target'):
        x = g['phase_days'] if g['phase_days'].notna().any() else g['date_obs']
        plt.errorbar(x, g['T_bb_K'], yerr=g['T_err_K'], marker='o', capsize=2, label=target)
    plt.ylabel('Blackbody temperature (K)')
    plt.xlabel('Days since discovery / obs date')
    plt.title('Continuum blackbody temperature estimate')
    plt.grid(alpha=0.2)
    plt.legend(fontsize=8)
    plt.show()

display(bb_df)

## 8. 单条谱线局部检查图

改 `CHECK_TARGET`、`CHECK_LINE` 和 `CHECK_INDEX` 来人工检查自动测量是否可信。

In [ ]:
CHECK_TARGET = summary.iloc[0]['target']
CHECK_LINE = default_lines_for_type(summary.iloc[0]['type'])[0] if summary.iloc[0]['type'] else 'SiII6355'
CHECK_INDEX = 0

items = sorted([s for s in spectra if s['target'] == CHECK_TARGET], key=lambda x: (pd.Timestamp.max if pd.isna(x['date_obs']) else x['date_obs']))
s = items[CHECK_INDEX]
z = s['z'] if np.isfinite(s['z']) else 0.0
wave_rest = s['wave'] / (1 + z)
flux = smooth_flux(s['flux'], preferred_window=21)
line = LINE_LIBRARY[CHECK_LINE]
rest = line['rest']
mask = (wave_rest > rest - LINE_HALF_WIDTH) & (wave_rest < rest + LINE_HALF_WIDTH)

result = measure_absorption_line(wave_rest, s['flux'], CHECK_LINE, LINE_HALF_WIDTH)

plt.figure(figsize=(10, 5))
plt.plot(wave_rest[mask], flux[mask], lw=0.9, label='smoothed spectrum')
plt.axvline(rest, color='red', ls='--', label=f'{CHECK_LINE} rest')
if result.get('status') == 'ok':
    plt.axvline(result['abs_wave'], color='green', ls=':', label='absorption minimum')
plt.title(f"{CHECK_TARGET} {Path(s['file']).name} - {CHECK_LINE}")
plt.xlabel('Rest wavelength (Angstrom)')
plt.ylabel(s.get('bunit', 'Flux'))
plt.grid(alpha=0.2)
plt.legend()
plt.show()

## 9. 保存 CSV 汇总

In [ ]:
if AUTO_SAVE:
    summary_path = OUTPUT_DIR / 'spectra_summary.csv'
    line_path = OUTPUT_DIR / 'line_diagnostics.csv'
    bb_path = OUTPUT_DIR / 'blackbody_temperature.csv'
    summary.to_csv(summary_path, index=False)
    line_df.to_csv(line_path, index=False)
    bb_df.to_csv(bb_path, index=False)
    print(f'Saved: {summary_path}')
    print(f'Saved: {line_path}')
    print(f'Saved: {bb_path}')
else:
    print('AUTO_SAVE=False; nothing written.')